# Task 4.1a: huấn luyện GCN và GGNN trên Graph A và Graph B

**Chuẩn bị:**
- Add Input dataset `DiverseVul 9077 - Processed Graph Data` (đầu ra của notebook `4_0_build_processed`).
- Settings, Accelerator: GPU T4 hoặc P100.

Notebook chạy thử nhanh trước (2 epoch, 200 mẫu mỗi split), sau đó chạy đầy đủ. Kết quả ghi liên tục vào `results_4_1a_gnn` nên phiên chết giữa chừng vẫn giữ được phần đã xong.

In [ ]:
import subprocess
import sys
from pathlib import Path

GIT_BRANCH = "main"
REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
REPO_DIR = Path("/tmp/g41a/software-bug-detection-using-graphs")  # Ngoài thư mục output
if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle.")
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "pull", "origin", GIT_BRANCH], cwd=REPO_DIR, check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO_DIR, text=True, capture_output=True).stdout)

In [ ]:
def run(cmd):
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode:
        raise RuntimeError(f"Lệnh thất bại với mã {proc.returncode}")

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Tìm dữ liệu đã xử lý

In [ ]:
found = [p.parent for p in Path("/kaggle/input").rglob("metadata.json")
         if (p.parent / "project_wise" / "vocab.json").is_file()]
assert found, "Không thấy dữ liệu đã xử lý, hãy Add Input dataset DiverseVul 9077 - Processed Graph Data."
PROCESSED = found[0]
OUT_DIR = Path("/kaggle/working/results_4_1a_gnn")
print("processed:", PROCESSED)

## Chạy thử nhanh (kiểm tra chạy được, không dùng để đánh giá)

In [ ]:
run([sys.executable, "scripts/train_gnn.py", "--processed-dir", str(PROCESSED),
     "--out-dir", "/tmp/g41a/smoke", "--protocols", "project_wise", "--seeds", "42",
     "--limit", "200", "--max-epochs", "2"])

## Chạy đầy đủ: 2 graph x 2 model x 2 protocol x 3 seed

In [ ]:
run([sys.executable, "scripts/train_gnn.py", "--processed-dir", str(PROCESSED),
     "--out-dir", str(OUT_DIR), "--seeds", "42", "43", "44"])

## Kết quả

Tệp đầy đủ trong `results_4_1a_gnn`: `results.csv`, `per_class.csv`, `confusion_matrices.json`, `training_history.json`, `summary.md`.

In [ ]:
print((OUT_DIR / "summary.md").read_text(encoding="utf-8"))